# 02 · Season exploration

What the data says before any model is fitted, and which of the model's assumptions it supports.
The reference season is **`fermedo_2024_25`**: the only complete one in the repo (16 teams, 240
matches, every pair home and away). `mansores_2025_26` is shown where its partial coverage still
says something.

The model in `season_prediction/03_model_evaluation` assumes:

1. goals for each side are **Poisson** (variance ≈ mean),
2. a team's scoring rate is its **attack** × the opponent's **defence weakness** × the league average,
3. a constant **home advantage**.

Each section below checks one of these.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())  # repo root from notebooks/<layer>/
sys.path.insert(0, str(ROOT / "src"))  # works without `pip install -e .`

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

from football_data_analysis.analyses.season_prediction.evaluation import fit_venue_factors, goals_dispersion
from football_data_analysis.core.league_table import compute_league_table
from football_data_analysis.core.pipeline import load_season
from football_data_analysis.analyses.season_prediction.team_strength import compute_team_strength

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

played, remaining = load_season("fermedo_2024_25")
print(f"fermedo_2024_25: {len(played)} played, {len(remaining)} remaining")

## 1. Final table

In [ ]:
table = compute_league_table(played)
table.index = range(1, len(table) + 1)
table

In [ ]:
# How the title race unfolded: cumulative points by matchweek, top 6 at the end highlighted.
long = pd.concat([
    played.assign(Team=played["Home Team"],
                  Pts=np.select([played["Home Goals"] > played["Away Goals"], played["Home Goals"] == played["Away Goals"]], [3, 1], 0)),
    played.assign(Team=played["Away Team"],
                  Pts=np.select([played["Away Goals"] > played["Home Goals"], played["Home Goals"] == played["Away Goals"]], [3, 1], 0)),
])
cum = long.pivot_table(index="Numeric Matchweek", columns="Team", values="Pts", aggfunc="sum").fillna(0).cumsum()
top = table["Team"].head(6).tolist()

fig, ax = plt.subplots(figsize=(9, 4.5))
for team in cum.columns:
    if team not in top:
        ax.plot(cum.index, cum[team], color="0.85", lw=1)
for team in top:
    ax.plot(cum.index, cum[team], lw=2, label=team)
ax.set(xlabel="Matchweek", ylabel="Points", title="Cumulative points, fermedo_2024_25 (top 6 coloured)")
ax.legend(frameon=False, fontsize=8)
plt.show()

## 2. Goals per match and the Poisson assumption

If goals were Poisson, variance / mean would be about 1. A ratio well above 1 (overdispersion)
means more blow-outs *and* more shut-outs than the model expects; the model's probabilities will
then be too confident in the middle of the distribution.

In [ ]:
print(f"Goals per match: {(played['Home Goals'] + played['Away Goals']).mean():.2f}")
goals_dispersion(played).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, col in zip(axes, ["Home Goals", "Away Goals"]):
    g = played[col]
    k = np.arange(0, g.max() + 1)
    observed = g.value_counts(normalize=True).reindex(k, fill_value=0)
    ax.bar(k, observed, color="#4C78A8", alpha=0.8, label="observed")
    ax.plot(k, poisson.pmf(k, g.mean()), "o-", color="#E45756", label=f"Poisson(λ={g.mean():.2f})")
    ax.set(title=col, xlabel="goals")
axes[0].set_ylabel("share of matches")
axes[0].legend(frameon=False)
plt.tight_layout(); plt.show()

## 3. Home advantage

The original model multiplies only the home side's expected goals by a fixed 1.15. The league
average it scales already *includes* home goals, so that adds goals rather than moving them
between venues. `fit_venue_factors` estimates a home and an away factor that reproduce the observed
means while keeping the total unchanged.

In [ ]:
home_f, away_f = fit_venue_factors(played)
res = played.assign(res=np.select([played["Home Goals"] > played["Away Goals"], played["Home Goals"] == played["Away Goals"]], ["H", "D"], "A"))
print(f"Mean goals  home {played['Home Goals'].mean():.2f}  away {played['Away Goals'].mean():.2f}")
print(f"Fitted venue factors: home {home_f:.3f}, away {away_f:.3f}  (fixed model: 1.15 / 1.00)")
res["res"].value_counts(normalize=True).reindex(["H", "D", "A"]).round(3).rename("share")

## 4. Team strength

Attack = goals scored per game ÷ league average; defence = league average ÷ (conceded per game +
0.1). Both are 1.0 for an average team, and a *higher* defence is better. The +0.1 keeps a team
that has conceded nothing from getting an infinite rating.

In [ ]:
strength = compute_team_strength(played)
st = strength.table.set_index("Team")
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(st["Attack Strength"], st["Defense Strength"], s=40, color="#4C78A8")
for team, r in st.iterrows():
    ax.annotate(team, (r["Attack Strength"], r["Defense Strength"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(1, color="0.7", lw=0.8); ax.axvline(1, color="0.7", lw=0.8)
ax.set(xlabel="attack strength (>1 scores more)", ylabel="defence strength (>1 concedes less)",
       title=f"Team strength (league avg {strength.league_avg_goals_per_game:.2f} goals per team per game)")
plt.show()

In [ ]:
# Does strength explain the table? Points vs attack/defence ratio.
ratio = (st["Attack Strength"] * st["Defense Strength"]).rename("attack × defence")
check = table.set_index("Team")[["Points"]].join(ratio)
print(f"Spearman rho, points vs strength: {check.corr(method='spearman').iloc[0, 1]:.3f}")
check.sort_values("Points", ascending=False).round(3)

## 5. The current season, with its coverage caveat

`mansores_2025_26` holds Mansores' own 22 fixtures only. The table below is exact for Mansores;
every other row reflects one match against Mansores, so the strengths are single-game estimates.

In [ ]:
m_played, m_remaining = load_season("mansores_2025_26")
m_table = compute_league_table(m_played)
m_table.index = range(1, len(m_table) + 1)
print(f"{len(m_played)} played, {len(m_remaining)} remaining (Mansores' fixtures only)")
m_table